# Forecast explainability

## Objective and method

Explain the persisted XGBoost champion with SHAP TreeExplainer at global, feature-family, horizon, and local SKU-store levels. This notebook is analysis-only: it reads the 2,000 out-of-sample explanation rows produced after final training and never fits a model.

TreeExplainer was selected because it computes model-specific additive attributions efficiently. Permutation importance was rejected as the primary method because correlated lag and rolling features can make its rankings unstable. SHAP values describe model behavior relative to its baseline; they do not estimate causal demand effects.


In [ ]:
profile = "dev"
run_id = "notebook-explainability"
shap_run_id = "calibrated-20260809"
force = False
execute_stage = False


In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

from retail_forecasting.config import load_config
from retail_forecasting.explainability import analyze_tree_shap

config = load_config(profile)
explanation_dir = config.paths.artifacts / "explainability" / shap_run_id
attribution_paths = sorted(explanation_dir.glob("*-shap-values.parquet"))
if not attribution_paths:
    raise FileNotFoundError(f"No tree SHAP values found in {explanation_dir}")
attributions = pd.read_parquet(attribution_paths[0])
analysis = analyze_tree_shap(attributions, top_n=20, local_n=10)
analysis.summary


## Global importance and direction

Mean absolute SHAP measures how strongly a feature moves predictions, in forecast-demand units. Mean signed SHAP and positive share show whether the sampled attributions usually raise or lower predictions relative to the model baseline. Cancellation makes signed importance unsuitable for ranking, so ranking uses absolute values.


In [ ]:
for image_path in sorted(explanation_dir.glob("*-shap-*.png")):
    display(Image(filename=str(image_path)))
display(
    analysis.global_importance[[
        "rank", "feature", "feature_group", "mean_abs_shap",
        "importance_share", "mean_shap", "positive_share",
    ]].style.format({
        "mean_abs_shap": "{:.3f}", "importance_share": "{:.1%}",
        "mean_shap": "{:+.3f}", "positive_share": "{:.1%}",
    })
)


## Feature families and horizon stability

A useful forecast explanation should remain intelligible across the complete 28-day horizon. The following views test whether identifiers, prices, or calendar fields dominate the model and whether the leading features change as the target date moves further from the origin.


In [ ]:
display(analysis.group_importance.style.format({"mean_abs_shap": "{:.3f}", "importance_share": "{:.1%}"}))
top_features = analysis.global_importance.head(5)["feature"].tolist()
horizon_profile = (
    analysis.horizon_importance.loc[analysis.horizon_importance["feature"].isin(top_features)]
    .pivot(index="horizon", columns="feature", values="mean_abs_shap")
)
display(horizon_profile.style.format("{:.3f}"))
axis = horizon_profile.plot(figsize=(12, 5), marker="o", title="Top feature importance by forecast horizon")
axis.set_ylabel("Mean absolute SHAP")
axis.set_xlabel("Forecast horizon")
axis.grid(axis="y", alpha=0.25)
display(
    analysis.segment_importance.style.format({
        "mean_total_abs_shap": "{:.3f}",
        "top_feature_mean_abs_shap": "{:.3f}",
    })
)


## Local SKU-store explanations

These are the sampled predictions with the largest total absolute attribution. A positive driver raises the forecast relative to the explainer baseline; a negative driver lowers it. Large local magnitudes identify cases for manual review, not necessarily model errors.


In [ ]:
display(
    analysis.local_explanations.style.format({
        "total_abs_shap": "{:.3f}", "top_driver_shap": "{:+.3f}",
    })
)


In [ ]:
summary = analysis.summary
global_by_feature = analysis.global_importance.set_index("feature")
rolling = global_by_feature.loc["rolling_mean_56"]
origin = global_by_feature.loc["origin_day"]
categories = analysis.segment_importance.query("segment_type == 'category'").set_index("segment")
stores = analysis.segment_importance.query("segment_type == 'store'").set_index("segment")
display(Markdown(f"""
## Interpretation and model risks

- The analysis covers **{summary['sample_rows']:,} rows**, **{summary['series_count']:,} SKU-store series**, all **{summary['horizon_count']} horizons**, and {summary['feature_count']} features.
- Demand-history variables account for **{summary['demand_history_share']:.1%}** of total mean absolute attribution. Calendar and horizon contribute {summary['calendar_horizon_share']:.1%}, identifiers {summary['identifier_share']:.1%}, and prices {summary['price_share']:.1%}.
- `{summary['top_feature']}` alone contributes **{summary['top_feature_share']:.1%}** and ranks first in **{summary['top_feature_horizon_count']}/{summary['horizon_count']} horizons**. The top five features concentrate **{summary['top_five_share']:.1%}** of attribution.
- `rolling_mean_56` lowers the prediction relative to baseline in **{1 - rolling['positive_share']:.1%}** of sampled rows, but the beeswarm shows that its high-value tail can add more than two demand units. This asymmetry is consistent with many intermittent low-demand rows and a smaller high-demand tail.
- `origin_day` is positive in **{origin['positive_share']:.1%}** of rows. Its global share is small, but the one-sided direction is a temporal-drift warning: later forecast origins systematically receive an uplift and should be monitored after deployment.
- Low identifier importance reduces, but does not eliminate, the risk that the global model mainly memorizes SKU or store encodings. Low price importance is plausible for M5's limited price/promotion signal, but it also shows why richer promotion and availability data are needed.
- HOBBIES has the largest category attribution magnitude at **{categories.loc['HOBBIES', 'mean_total_abs_shap']:.3f}**, versus {categories.loc['FOODS', 'mean_total_abs_shap']:.3f} for FOODS. WI_2 is the largest store at **{stores.loc['WI_2', 'mean_total_abs_shap']:.3f}**. These values measure how far the model moves from baseline, not predictive error.

The strong concentration in smoothed demand history explains why the model is stable at store level but struggles with SKU-day spikes: it learns demand level well and exact timing poorly.
"""))


## Limitations and leakage review

All rolling and lag inputs were generated at the forecast origin, so no target from the 28-day evaluation window is consumed. `origin_day` is known at prediction time but can encode temporal drift rather than a durable business relationship. Calendar, SNAP, and planned prices are also known-future covariates.

The historical champion artifact stores attribution values but not the transformed feature values or SHAP baseline, which prevents exact dependence and waterfall plots for this already-completed run. The production writer now persists `feature_*`, `base_value`, and `model_output` fields for future explanations. Recomputing SHAP for this champion would not require retraining, but the quantitative global, horizon, and local attribution analysis above is complete from the stored values.
